In [1]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import scipy.stats

pd.options.display.max_colwidth = 999
pd.options.display.max_columns = 999

In [2]:
ds = load_dataset("open-llm-leaderboard/contents")
benchmark_df = ds['train'].to_pandas()

In [3]:
all_models = ["meta-llama/Meta-Llama-3-8B-Instruct", "meta-llama/Llama-3.2-3B-Instruct", "mistralai/Mistral-7B-Instruct-v0.2", "Qwen/Qwen2.5-7B-Instruct", "tiiuae/falcon-7b-instruct", \
                             "Qwen/Qwen2.5-14B-Instruct", "Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-0.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct"]

benchmark_df = benchmark_df[benchmark_df.fullname.str.contains("|".join(all_models))]

In [4]:
benchmark_df = benchmark_df.groupby(['fullname'])[['IFEval','BBH','MMLU-PRO']].max().reset_index(drop=False)

In [5]:
benchmark_df = benchmark_df[benchmark_df.fullname != 'Qwen/Qwen2.5-14B-Instruct-1M'].reset_index(drop=True)
benchmark_df = benchmark_df[benchmark_df.fullname != 'Qwen/Qwen2.5-7B-Instruct-1M'].reset_index(drop=True)
benchmark_df = benchmark_df[benchmark_df.fullname != 'tiiuae/falcon-7b-instruct'].reset_index(drop=True)
benchmark_df = benchmark_df[benchmark_df.fullname.str.contains("Phi") == False].reset_index(drop=True)

In [6]:
benchmark_df

,fullname,IFEval,BBH,MMLU-PRO
0,Qwen/Qwen2.5-0.5B-Instruct,31.529121,8.434864,7.995346
1,Qwen/Qwen2.5-1.5B-Instruct,44.755693,19.809786,19.991135
2,Qwen/Qwen2.5-14B-Instruct,81.577769,48.360707,43.382462
3,Qwen/Qwen2.5-3B-Instruct,64.749199,25.801394,25.051714
4,Qwen/Qwen2.5-7B-Instruct,75.852516,34.892117,36.521129
5,meta-llama/Llama-3.2-3B-Instruct,73.931613,24.059186,24.386820
6,meta-llama/Meta-Llama-3-8B-Instruct,74.083986,28.244950,29.604388
7,mistralai/Mistral-7B-Instruct-v0.2,54.962278,22.910602,19.076906


In [7]:
lpp_df = pd.read_csv("LPP.csv").dropna(subset=['effective_rank']).reset_index(drop=True)
lpp_df = lpp_df[lpp_df.prefix_tokens > 10].reset_index(drop=True)
lpp_df = lpp_df[lpp_df.model_id.str.contains("|".join(all_models))]

In [8]:
lpp_df = lpp_df.groupby(['model_id'])[['mean_next_token_entropy','participation_ratio','effective_rank']].agg(['min','max','mean'])
lpp_df = lpp_df.reset_index(drop=False)
lpp_df.columns = [i[0]+"_"+ str(i[1]) for i in lpp_df.columns]
lpp_df = lpp_df.rename(columns={'model_id_': 'model_id'})

In [9]:
lpp_df

,model_id,mean_next_token_entropy_min,mean_next_token_entropy_max,mean_next_token_entropy_mean,participation_ratio_min,participation_ratio_max,participation_ratio_mean,effective_rank_min,effective_rank_max,effective_rank_mean
0,Qwen/Qwen2.5-0.5B-Instruct,0.115086,0.220743,0.154024,0.011798,0.031864,0.022436,0.018652,0.135234,0.089889
1,Qwen/Qwen2.5-1.5B-Instruct,0.093120,0.191280,0.125428,0.007134,0.013899,0.011162,0.011849,0.082870,0.056579
2,Qwen/Qwen2.5-14B-Instruct,0.019099,0.177340,0.106832,0.003689,0.007315,0.004455,0.010552,0.023769,0.017999
3,Qwen/Qwen2.5-3B-Instruct,0.056345,0.159431,0.113601,0.005684,0.009691,0.007867,0.010229,0.064886,0.043623
4,Qwen/Qwen2.5-7B-Instruct,0.087032,0.166075,0.113909,0.005484,0.007716,0.006953,0.010736,0.043613,0.032333
5,meta-llama/Llama-3.2-3B-Instruct,0.079830,0.179076,0.125702,0.005912,0.032475,0.020287,0.009199,0.181351,0.108232
6,meta-llama/Meta-Llama-3-8B-Instruct,0.054458,0.147321,0.092300,0.006385,0.023997,0.015685,0.009202,0.122512,0.073987
7,mistralai/Mistral-7B-Instruct-v0.2,0.043618,0.186277,0.096328,0.004985,0.020603,0.011786,0.009902,0.130520,0.066051


In [10]:
benchmark_df1 = pd.merge(benchmark_df, lpp_df, left_on=['fullname'], right_on=['model_id'], how='inner')


In [13]:
extrinsic_tasks = ['IFEval','BBH','MMLU-PRO']

In [14]:
intrinsic_tasks = [
 'mean_next_token_entropy_min',
 'effective_rank_max',
 'participation_ratio_max']

In [17]:
lpp1_res = pd.DataFrame()
i = 0
for col1 in extrinsic_tasks:
    for col2 in intrinsic_tasks:
        test = scipy.stats.pearsonr(benchmark_df1[col1], benchmark_df1[col2])
        lpp1_res.loc[i, "intrinsic"] = col2
        lpp1_res.loc[i, "extrinsic"] = col1
        lpp1_res.loc[i, "correlation"] = test.statistic
        lpp1_res.loc[i, "pvalue"] = test.pvalue

        i += 1
        
        if test.pvalue <= 0.05:
            print (col1, col2, test)

    print ("===============================")

BBH mean_next_token_entropy_min PearsonRResult(statistic=-0.7531253031757476, pvalue=0.030994854645756815)


In [18]:
dummy_tasks1 = pd.read_csv("spc.csv")
dummy_tasks1.columns = ['fullname','spc_em','spc_f1']

dummy_tasks2 = pd.read_csv("ar.csv")
dummy_tasks2.columns = ['fullname','ar_acc','ar_overconf','ar_underconf']

In [19]:
benchmark_df1 = pd.merge(benchmark_df1, dummy_tasks1, how='left')
benchmark_df1 = pd.merge(benchmark_df1, dummy_tasks2, how='left')

In [20]:
benchmark_df1 = benchmark_df1.dropna().reset_index(drop=True)

In [22]:
benchmark_df1[['IFEval','BBH','MMLU-PRO']].corr()

,IFEval,BBH,MMLU-PRO
IFEval,1.000000,0.862344,0.903535
BBH,0.862344,1.000000,0.979442
MMLU-PRO,0.903535,0.979442,1.000000


In [23]:
new_tasks = ['spc_f1','ar_acc']

In [24]:
lpp3_res = pd.DataFrame()
i = 0

for col1 in new_tasks:
    for col2 in intrinsic_tasks:
        test = scipy.stats.pearsonr(benchmark_df1[col1], benchmark_df1[col2])
        lpp3_res.loc[i, "intrinsic"] = col2
        lpp3_res.loc[i, "extrinsic"] = col1
        lpp3_res.loc[i, "correlation"] = test.statistic
        lpp3_res.loc[i, "pvalue"] = test.pvalue

        i += 1
        
        if test.pvalue <= 0.05:
            print (col1, col2, test)

    print ("===============================")

spc_f1 effective_rank_max PearsonRResult(statistic=-0.7134903087299195, pvalue=0.04688684607461225)
spc_f1 participation_ratio_max PearsonRResult(statistic=-0.7942540767379044, pvalue=0.01855213276594959)
ar_acc mean_next_token_entropy_min PearsonRResult(statistic=-0.7896850498254326, pvalue=0.01974268812413371)
